# SafeEat YOLOv5 Training Notebook — Public Clean Version

This notebook documents the model-training workflow used for the SafeEat fruit-quality detection project.

It is designed for a public GitHub portfolio repository. It intentionally does **not** contain:

- Roboflow API keys
- Google Drive file IDs
- private Colab links
- personal filesystem paths
- backend server IP addresses
- raw dataset files

Use this notebook as a reproducible template: add your own credentials through Colab secrets or environment variables, download datasets under their licenses, then run the preprocessing, training, validation, and export cells.


## 1. Environment setup

The original workflow used Ultralytics YOLOv5 in Google Colab and exported the trained model to TensorFlow Lite for mobile usage.


In [ ]:
# Recommended runtime: Google Colab with GPU enabled
# Runtime > Change runtime type > GPU

!nvidia-smi || true


In [ ]:
from pathlib import Path
import os
import random
import shutil
import yaml
from collections import defaultdict

RANDOM_SEED = 42
random.seed(RANDOM_SEED)

WORK_DIR = Path('/content')
RAW_DATA_DIR = WORK_DIR / 'raw_roboflow_datasets'
COMBINED_DIR = WORK_DIR / 'combined_dataset'
SPLIT_DIR = WORK_DIR / 'dataset_split'
YOLOV5_DIR = WORK_DIR / 'yolov5'

for path in [RAW_DATA_DIR, COMBINED_DIR, SPLIT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

FINAL_CLASS_NAMES = [
    'fresh_apple',
    'rotten_apple',
    'fresh_banana',
    'rotten_banana',
    'fresh_orange',
    'rotten_orange',
]

print('Target classes:', FINAL_CLASS_NAMES)


## 2. Install YOLOv5 and Roboflow client

Roboflow is only needed if you want to re-download the source datasets. If you already have YOLO-format datasets locally, skip the download cell and place them under `/content/raw_roboflow_datasets/`.


In [ ]:
!git clone https://github.com/ultralytics/yolov5.git /content/yolov5
%cd /content/yolov5
%pip install -qr requirements.txt
%pip install -q roboflow

import torch
print('Torch version:', torch.__version__)


## 3. Dataset sources

Raw datasets are not committed to the SafeEat GitHub repository. Cite dataset sources in `docs/DATASET.md` and download them only under their respective licenses.

The original experiment combined multiple Roboflow Universe datasets, filtered the labels, relabelled selected classes into a 6-class fruit-quality task, split the combined data into train/validation/test folders, trained YOLOv5, and exported the model to TFLite.

Edit the `class_mapping` values if the downloaded dataset versions expose different class IDs in their `data.yaml` files.


In [ ]:
# Public dataset references used in the project documentation.
# class_mapping maps original YOLO class IDs -> final SafeEat class IDs.
# Verify each mapping against the downloaded dataset's data.yaml before training.

DATASET_SOURCES = [
    {
        'name': 'fruits_hnsor_v3',
        'workspace': 'nivin-a',
        'project': 'fruits-hnsor',
        'version': 3,
        'class_mapping': {0: 0, 1: 1, 3: 2, 5: 3, 6: 4, 7: 5},
    },
    {
        'name': 'applebapples_v2',
        'workspace': 'applesbob',
        'project': 'applebapples',
        'version': 2,
        'class_mapping': {0: 1, 1: 3},
    },
    {
        'name': 'fruits_detection_quality_v9',
        'workspace': 'jawaharlal-nehru-university-utour',
        'project': 'fruits-detection-and-quality-analysis',
        'version': 9,
        'class_mapping': {0: 1, 1: 3, 4: 0, 5: 4, 16: 2, 17: 5},
    },
    # Optional additional sources documented in docs/DATASET.md.
    # Add them here after verifying their class IDs and licenses.
]

for source in DATASET_SOURCES:
    print(f"{source['name']}: {source['workspace']}/{source['project']} v{source['version']}")


## 4. Download datasets safely

This cell uses a runtime secret, not a hard-coded API key. In Colab, store your key as a secret named `ROBOFLOW_API_KEY`, or enter it at runtime when prompted.


In [ ]:
DOWNLOAD_FROM_ROBOFLOW = False  # Change to True when you want to re-download the datasets.

if DOWNLOAD_FROM_ROBOFLOW:
    try:
        from google.colab import userdata
        ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
    except Exception:
        ROBOFLOW_API_KEY = os.environ.get('ROBOFLOW_API_KEY')

    if not ROBOFLOW_API_KEY:
        from getpass import getpass
        ROBOFLOW_API_KEY = getpass('Enter your Roboflow API key: ')

    from roboflow import Roboflow
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)

    for source in DATASET_SOURCES:
        project = rf.workspace(source['workspace']).project(source['project'])
        version = project.version(source['version'])
        dataset = version.download('yolov5', location=str(RAW_DATA_DIR / source['name']))
        print('Downloaded:', source['name'], '->', dataset.location)
else:
    print('Skipping Roboflow download. Set DOWNLOAD_FROM_ROBOFLOW=True to download datasets.')


## 5. Preprocess: filter, relabel, and combine YOLO annotations

YOLO label rows use this format:

```text
class_id x_center y_center width height
```

The function below keeps only selected classes, remaps them to the SafeEat 6-class label space, and copies the corresponding images. File names are prefixed with the dataset name to avoid collisions when combining datasets.


In [ ]:
IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png', '.webp'}


def reset_dir(path: Path):
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


def find_image_for_label(images_dir: Path, label_filename: str):
    stem = Path(label_filename).stem
    for suffix in IMAGE_SUFFIXES:
        candidate = images_dir / f'{stem}{suffix}'
        if candidate.exists():
            return candidate
    return None


def copy_filtered_split(dataset_name: str, split_name: str, images_dir: Path, labels_dir: Path, class_mapping: dict, out_images_dir: Path, out_labels_dir: Path):
    copied = 0
    skipped_empty = 0
    missing_images = 0

    if not labels_dir.exists() or not images_dir.exists():
        print(f'Skip missing split: {dataset_name}/{split_name}')
        return copied, skipped_empty, missing_images

    out_images_dir.mkdir(parents=True, exist_ok=True)
    out_labels_dir.mkdir(parents=True, exist_ok=True)

    for label_path in sorted(labels_dir.glob('*.txt')):
        new_lines = []
        with open(label_path, 'r', encoding='utf-8') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                old_class_id = int(float(parts[0]))
                if old_class_id in class_mapping:
                    parts[0] = str(class_mapping[old_class_id])
                    new_lines.append(' '.join(parts))

        if not new_lines:
            skipped_empty += 1
            continue

        image_path = find_image_for_label(images_dir, label_path.name)
        if image_path is None:
            missing_images += 1
            continue

        safe_prefix = f'{dataset_name}_{split_name}'
        new_stem = f'{safe_prefix}_{label_path.stem}'
        new_image_path = out_images_dir / f'{new_stem}{image_path.suffix.lower()}'
        new_label_path = out_labels_dir / f'{new_stem}.txt'

        shutil.copy2(image_path, new_image_path)
        with open(new_label_path, 'w', encoding='utf-8') as f:
            f.write('\n'.join(new_lines) + '\n')
        copied += 1

    return copied, skipped_empty, missing_images


def combine_datasets(dataset_sources, raw_data_dir: Path, combined_dir: Path):
    reset_dir(combined_dir)
    out_images_dir = combined_dir / 'images'
    out_labels_dir = combined_dir / 'labels'
    out_images_dir.mkdir(parents=True, exist_ok=True)
    out_labels_dir.mkdir(parents=True, exist_ok=True)

    report = []
    for source in dataset_sources:
        dataset_dir = raw_data_dir / source['name']
        for split_name in ['train', 'valid', 'test']:
            images_dir = dataset_dir / split_name / 'images'
            labels_dir = dataset_dir / split_name / 'labels'
            copied, skipped_empty, missing_images = copy_filtered_split(
                source['name'], split_name, images_dir, labels_dir, source['class_mapping'], out_images_dir, out_labels_dir
            )
            report.append({
                'dataset': source['name'],
                'split': split_name,
                'copied': copied,
                'skipped_empty': skipped_empty,
                'missing_images': missing_images,
            })
    return report

# Run after datasets have been downloaded or manually placed under RAW_DATA_DIR.
# report = combine_datasets(DATASET_SOURCES, RAW_DATA_DIR, COMBINED_DIR)
# report


## 6. Check class distribution

This sanity check helps identify class imbalance before training.


In [ ]:
def count_yolo_instances(labels_dir: Path):
    counts = defaultdict(int)
    files = sorted(labels_dir.glob('*.txt'))
    for label_path in files:
        with open(label_path, 'r', encoding='utf-8') as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    counts[int(float(parts[0]))] += 1
    return dict(sorted(counts.items()))

# counts = count_yolo_instances(COMBINED_DIR / 'labels')
# for class_id, count in counts.items():
#     print(class_id, FINAL_CLASS_NAMES[class_id], count)


## 7. Split combined dataset into train/validation/test

The original notebook used a 70/20/10 split.


In [ ]:
def split_dataset(combined_dir: Path, split_dir: Path, train_ratio=0.70, valid_ratio=0.20, test_ratio=0.10):
    assert abs(train_ratio + valid_ratio + test_ratio - 1.0) < 1e-6

    reset_dir(split_dir)
    image_files = [p for p in (combined_dir / 'images').iterdir() if p.suffix.lower() in IMAGE_SUFFIXES]
    image_files = sorted(image_files)
    random.shuffle(image_files)

    n_total = len(image_files)
    n_train = int(n_total * train_ratio)
    n_valid = int(n_total * valid_ratio)

    splits = {
        'train': image_files[:n_train],
        'valid': image_files[n_train:n_train + n_valid],
        'test': image_files[n_train + n_valid:],
    }

    for split_name, files in splits.items():
        images_out = split_dir / split_name / 'images'
        labels_out = split_dir / split_name / 'labels'
        images_out.mkdir(parents=True, exist_ok=True)
        labels_out.mkdir(parents=True, exist_ok=True)

        for image_path in files:
            label_path = combined_dir / 'labels' / f'{image_path.stem}.txt'
            if not label_path.exists():
                continue
            shutil.copy2(image_path, images_out / image_path.name)
            shutil.copy2(label_path, labels_out / label_path.name)

    return {split_name: len(files) for split_name, files in splits.items()}

# split_summary = split_dataset(COMBINED_DIR, SPLIT_DIR)
# split_summary


## 8. Create `data.yaml`

YOLOv5 uses `data.yaml` to locate the dataset and define class names.


In [ ]:
def write_data_yaml(split_dir: Path, class_names):
    data = {
        'path': str(split_dir),
        'train': 'train/images',
        'val': 'valid/images',
        'test': 'test/images',
        'nc': len(class_names),
        'names': class_names,
    }
    yaml_path = split_dir / 'data.yaml'
    with open(yaml_path, 'w', encoding='utf-8') as f:
        yaml.safe_dump(data, f, sort_keys=False)
    return yaml_path

# data_yaml = write_data_yaml(SPLIT_DIR, FINAL_CLASS_NAMES)
# print(data_yaml.read_text())


## 9. Train YOLOv5

The original training used YOLOv5s, image size 640, and 30 epochs. Adjust batch size depending on your GPU memory.


In [ ]:
%cd /content/yolov5

# Uncomment after preparing /content/dataset_split/data.yaml
# !python train.py \
#     --img 640 \
#     --epochs 30 \
#     --batch 16 \
#     --data /content/dataset_split/data.yaml \
#     --weights yolov5s.pt \
#     --project runs/train \
#     --name safeeat_yolov5


## 10. Validate the trained model

Save the metrics, confusion matrix, and example prediction images for your GitHub README or project report.


In [ ]:
%cd /content/yolov5

# BEST_WEIGHTS = '/content/yolov5/runs/train/safeeat_yolov5/weights/best.pt'
# !python val.py --weights $BEST_WEIGHTS --data /content/dataset_split/data.yaml --img 640


## 11. Export to TensorFlow Lite

The Android project can use TFLite assets, and the deployed backend can also use the trained weights depending on your inference architecture.


In [ ]:
%cd /content/yolov5

# BEST_WEIGHTS = '/content/yolov5/runs/train/safeeat_yolov5/weights/best.pt'
# !python export.py --weights $BEST_WEIGHTS --include tflite --img 640

# Expected output example:
# /content/yolov5/runs/train/safeeat_yolov5/weights/best-fp16.tflite


## 12. Run demo inference

Place a few test images in `/content/demo_images/` and run YOLOv5 detection to generate example outputs.


In [ ]:
%cd /content/yolov5

# DEMO_IMAGES = '/content/demo_images'
# EXPORTED_WEIGHTS = '/content/yolov5/runs/train/safeeat_yolov5/weights/best.pt'
# !python detect.py --weights $EXPORTED_WEIGHTS --img 640 --conf 0.25 --source $DEMO_IMAGES


## 13. Export artifacts for the Android app

Copy the exported model into the Android project only when the file is small enough and the license/project policy allows redistribution.

For GitHub portfolio use, it is often enough to include:

- app source code
- TFLite model asset if it is allowed to publish
- `docs/DATASET.md`
- redacted Colab PDF
- this cleaned notebook
- training summary and screenshots


In [ ]:
# Example: download the exported TFLite file from Colab.
# from google.colab import files
# files.download('/content/yolov5/runs/train/safeeat_yolov5/weights/best-fp16.tflite')


## 14. GitHub publication checklist

Before committing notebooks or training artifacts:

- Restart runtime and run only public-safe cells.
- Clear all cell outputs if they contain local paths, URLs, IDs, or tokens.
- Do not commit API keys, Firebase config, Google Drive IDs, or private server addresses.
- Keep raw datasets out of the repository unless their licenses explicitly allow redistribution.
- Attribute Roboflow/dataset sources in `docs/DATASET.md`.
